# Smart Shepherd: TFLite Micro Anomaly Detection
### 1D-CNN Autoencoder for ESP32 Collars

This notebook implements the unsupervised anomaly detection pipeline for on-device inference.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# 1. Data Generation (Healthy Baseline)
def generate_healthy(n=50000):
    # [temp, bpm, spo2, ax, ay, az]
    temp = np.random.normal(39.0, 0.2, (n, 12, 1))
    bpm = np.random.normal(75, 5, (n, 12, 1))
    spo2 = np.random.normal(98, 0.5, (n, 12, 1))
    accel = np.random.normal(0, 0.05, (n, 12, 3))
    X = np.concatenate([temp, bpm, spo2, accel], axis=-1)
    # Normalize
    X[:,:,0] = (X[:,:,0]-35)/10
    X[:,:,1] = X[:,:,1]/200
    X[:,:,2] = X[:,:,2]/100
    return X.astype(np.float32)

X_train = generate_healthy()
print(f"Training data shape: {X_train.shape}")

### 2. Model Architecture
Constraint: < 8,000 parameters

In [ ]:
def build_model():
    model = models.Sequential([
        layers.Input(shape=(12, 6)),
        layers.Conv1D(16, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Conv1D(8, 3, activation='relu', padding='same'),
        layers.UpSampling1D(2),
        layers.Conv1D(16, 3, activation='relu', padding='same'),
        layers.Conv1D(6, 3, activation='sigmoid', padding='same')
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

model = build_model()
model.summary()

### 3. Training & Quantization

In [ ]:
model.fit(X_train, X_train, epochs=10, batch_size=64)

# INT8 Quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
def rep_gen():
    for i in range(100):
        yield [X_train[i:i+1]]
converter.representative_dataset = rep_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
tflite_model = converter.convert()

with open("model_quantized.tflite", "wb") as f:
    f.write(tflite_model)
print(f"TFLite Model Size: {len(tflite_model)/1024:.2f} KB")